<a href="https://colab.research.google.com/github/bitabaroutian/Multi-scale-cross-attention-for-multi-class-imbalanced-image-classification/blob/main/Multi_scale_cross_attention_for_multi_class_imbalanced_image_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q scikit-learn tqdm
import os, csv, json, time, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from tqdm.auto import tqdm

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU only')

torch: 2.14.1+cu130
cuda available: True
GPU: Tesla T4


In [11]:
CONFIG = {
    'dataset': 'cifar10',
    'data_root': './data',
    'fer_csv': None,
    'rho': None,
    'ablation': 'pnmscca',
    'lr': 1e-3,
    'batch_size': 1024,
    'epochs': 25,
    'nf': 15,
    'num_workers': 0,
    'seed': 0,
    'checkpoint_dir': './checkpoints',
}

def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print(CONFIG)

Device: cuda
{'dataset': 'cifar10', 'data_root': './data', 'fer_csv': None, 'rho': None, 'ablation': 'pnmscca', 'lr': 0.001, 'batch_size': 1024, 'epochs': 25, 'nf': 15, 'num_workers': 0, 'seed': 0, 'checkpoint_dir': './checkpoints'}


In [12]:
def class_counts(labels, num_classes):
    labels_arr = np.asarray(labels, dtype=np.int64)
    return np.bincount(labels_arr, minlength=num_classes).astype(np.int64)

def imbalance_ratio(counts):
    positive = counts[counts > 0]
    return float(positive.max() / positive.min()) if len(positive) else float('nan')

def _subsample_class(indices, keep_count, rng):
    if keep_count >= len(indices):
        return indices
    return np.sort(rng.choice(indices, size=keep_count, replace=False))

def make_imbalanced_indices(labels, num_classes, minority_classes, majority_classes,
                            rho, seed=0, prekeep_fraction=1.0):
    rng = np.random.default_rng(seed)
    labels_arr = np.asarray(labels, dtype=np.int64)
    all_indices = np.arange(len(labels_arr))
    selected = []
    for cls in range(num_classes):
        cls_idx = all_indices[labels_arr == cls]
        if prekeep_fraction < 1.0:
            keep = max(1, int(round(len(cls_idx) * prekeep_fraction)))
            cls_idx = _subsample_class(cls_idx, keep, rng)
        selected.append(cls_idx)
    ref_majority = min(len(selected[c]) for c in majority_classes)
    target_minority = max(1, int(round(ref_majority / float(rho))))
    final_parts = []
    for cls in range(num_classes):
        cls_idx = selected[cls]
        if cls in minority_classes:
            cls_idx = _subsample_class(cls_idx, target_minority, rng)
        elif cls not in majority_classes:
            raise ValueError(f'Class {cls} is neither minority nor majority')
        final_parts.append(cls_idx)
    indices = np.sort(np.concatenate(final_parts))
    counts = class_counts(labels_arr[indices], num_classes)
    return indices, {
        'counts': counts,
        'rho_effective': imbalance_ratio(counts),
        'rho_target': rho,
        'minority_classes': list(minority_classes),
        'majority_classes': list(majority_classes),
        'prekeep_fraction': prekeep_fraction,
        'num_samples': int(len(indices)),
    }

def dataset_group_config(dataset):
    name = dataset.lower()
    if name == 'svhn':
        return {'num_classes': 10, 'majority_classes': [0,1,2,3,4], 'minority_classes': [5,6,7,8,9],
                'prekeep_fraction': 0.25, 'default_rho': 14.87, 'image_size': 32}
    if name in {'cifar10', 'cifar-10'}:
        return {'num_classes': 10, 'majority_classes': [5,6,7,8,9], 'minority_classes': [0,1,2,3,4],
                'prekeep_fraction': 1.0, 'default_rho': 10.0, 'image_size': 32}
    if name == 'fer2013':
        return {'num_classes': 7, 'majority_classes': [0,2,3,4,5,6], 'minority_classes': [1],
                'prekeep_fraction': 1.0, 'default_rho': 16.54, 'image_size': 32, 'natural_imbalance': True}
    raise ValueError(f'Unsupported dataset: {dataset}')

print('OK: imbalance helpers')

OK: imbalance helpers


In [13]:
class IndexedDataset(Dataset):
    def __init__(self, base):
        self.base = base
    def __len__(self):
        return len(self.base)
    def __getitem__(self, index):
        image, label = self.base[index]
        label = torch.tensor(label, dtype=torch.long) if not torch.is_tensor(label) else label.long()
        return image, label

class FER2013CSV(Dataset):
    def __init__(self, csv_path, split='train', image_size=32, transform=None):
        self.transform, self.image_size = transform, image_size
        self.images, self.labels = [], []
        split_map = {'train': {'training'}, 'test': {'publictest', 'privatetest', 'test'}}
        allowed = split_map.get(split.lower(), {split.lower()})
        with open(csv_path, 'r', newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            has_usage = 'Usage' in (reader.fieldnames or [])
            for row in reader:
                if has_usage:
                    usage = row['Usage'].strip().lower().replace(' ', '')
                    if usage not in allowed:
                        continue
                elif split.lower() != 'train':
                    continue
                pixels = np.fromstring(row['pixels'], sep=' ', dtype=np.float32)
                self.images.append(pixels.reshape(48, 48))
                self.labels.append(int(row['emotion']))
        if not self.images:
            raise RuntimeError(f'No FER2013 samples for split={split}')
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        tensor = torch.from_numpy(self.images[index]).unsqueeze(0) / 255.0
        tensor = tensor.repeat(3, 1, 1)
        tensor = torch.nn.functional.interpolate(
            tensor.unsqueeze(0), size=(self.image_size, self.image_size),
            mode='bilinear', align_corners=False).squeeze(0)
        if self.transform is not None:
            tensor = self.transform(tensor)
        return tensor, torch.tensor(self.labels[index], dtype=torch.long)

def _default_transform(image_size=32):
    return transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor()])

def _extract_labels(dataset):
    if hasattr(dataset, 'labels'):
        labels = dataset.labels
        return labels.cpu().numpy().astype(np.int64) if torch.is_tensor(labels) else np.asarray(labels, dtype=np.int64)
    if hasattr(dataset, 'targets'):
        targets = dataset.targets
        return targets.cpu().numpy().astype(np.int64) if torch.is_tensor(targets) else np.asarray(targets, dtype=np.int64)
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

def load_base_dataset(name, root, train, image_size=32, fer_csv=None):
    name, root = name.lower(), Path(root)
    transform = _default_transform(image_size)
    if name == 'svhn':
        return IndexedDataset(datasets.SVHN(str(root), split='train' if train else 'test', download=True, transform=transform))
    if name in {'cifar10', 'cifar-10'}:
        return IndexedDataset(datasets.CIFAR10(str(root), train=train, download=True, transform=transform))
    if name == 'fer2013':
        return FER2013CSV(fer_csv or (root / 'fer2013.csv'), split='train' if train else 'test', image_size=image_size)
    raise ValueError(name)

def build_imbalanced_subset(dataset, dataset_name, rho=None, seed=0):
    cfg = dataset_group_config(dataset_name)
    labels = _extract_labels(dataset.base if isinstance(dataset, IndexedDataset) else dataset)
    if cfg.get('natural_imbalance'):
        indices = np.arange(len(labels))
        counts = np.bincount(labels, minlength=int(cfg['num_classes']))
        meta = {
            'counts': counts,
            'rho_effective': float(counts.max() / max(1, counts[counts > 0].min())),
            'rho_target': cfg['default_rho'],
            'minority_classes': cfg['minority_classes'],
            'majority_classes': cfg['majority_classes'],
            'prekeep_fraction': 1.0,
            'num_samples': int(len(indices)),
            'natural_imbalance': True,
        }
        return Subset(dataset, indices.tolist()), meta
    target_rho = float(rho if rho is not None else cfg['default_rho'])
    indices, meta = make_imbalanced_indices(
        labels, int(cfg['num_classes']), cfg['minority_classes'], cfg['majority_classes'],
        rho=target_rho, seed=seed, prekeep_fraction=float(cfg['prekeep_fraction']))
    return Subset(dataset, indices.tolist()), meta

def build_representatives(dataset, minority_classes, majority_classes, num_channels=3):
    class_sum, class_count = {}, {}
    for images, labels in DataLoader(dataset, batch_size=256, shuffle=False, num_workers=0):
        for cls in labels.unique().tolist():
            mask = labels == cls
            selected = images[mask]
            if cls not in class_sum:
                class_sum[cls], class_count[cls] = selected.sum(dim=0), int(mask.sum())
            else:
                class_sum[cls] = class_sum[cls] + selected.sum(dim=0)
                class_count[cls] += int(mask.sum())
    p = torch.cat([class_sum[c] / class_count[c] for c in minority_classes], dim=0).unsqueeze(0)
    n = torch.cat([class_sum[c] / class_count[c] for c in majority_classes], dim=0).unsqueeze(0)
    return p, n

def build_dataloaders(dataset_name, data_root='./data', rho=None, batch_size=1024,
                      num_workers=2, seed=0, fer_csv=None):
    cfg = dataset_group_config(dataset_name)
    image_size = int(cfg['image_size'])
    train_base = load_base_dataset(dataset_name, data_root, True, image_size, fer_csv)
    test_base = load_base_dataset(dataset_name, data_root, False, image_size, fer_csv)
    train_set, train_meta = build_imbalanced_subset(train_base, dataset_name, rho=rho, seed=seed)
    test_set, test_meta = build_imbalanced_subset(test_base, dataset_name, rho=rho, seed=seed + 1)
    positive_cube, negative_cube = build_representatives(train_set, cfg['minority_classes'], cfg['majority_classes'])
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                              pin_memory=torch.cuda.is_available())
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                             pin_memory=torch.cuda.is_available())
    meta = {
        'dataset': dataset_name, 'config': cfg, 'train': train_meta, 'test': test_meta,
        'pos_channels': int(positive_cube.shape[1]), 'neg_channels': int(negative_cube.shape[1]),
        'image_size': image_size, 'num_classes': int(cfg['num_classes']),
    }
    return train_loader, test_loader, meta, positive_cube, negative_cube

print('OK: datasets')

OK: datasets


In [14]:
SCALES = (1, 3, 5)

def _same_pad_conv2d(in_channels, out_channels, kernel_size, bias=True):
    return nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size,
                     stride=1, padding=kernel_size // 2, bias=bias)

class MultiScaleCrossAttention(nn.Module):
    def __init__(self, in_channels, pos_channels, neg_channels, nf=15, scales=SCALES,
                 use_positive=True, use_negative=True):
        super().__init__()
        self.nf, self.scales = nf, tuple(scales)
        self.use_positive, self.use_negative = use_positive, use_negative
        self.sqrt_d = float(nf) ** 0.5
        self.key_convs, self.value_convs = nn.ModuleDict(), nn.ModuleDict()
        self.pos_query_convs, self.neg_query_convs = nn.ModuleDict(), nn.ModuleDict()
        for s in self.scales:
            k = str(s)
            self.key_convs[k] = _same_pad_conv2d(in_channels, nf, s)
            self.value_convs[k] = _same_pad_conv2d(in_channels, nf, s)
            if use_positive:
                self.pos_query_convs[k] = _same_pad_conv2d(pos_channels, nf, s)
            if use_negative:
                self.neg_query_convs[k] = _same_pad_conv2d(neg_channels, nf, s)

    def _attention(self, key, value, query):
        return value * torch.softmax((key * query) / self.sqrt_d, dim=1)

    @staticmethod
    def _broadcast_cube(cube, batch_size):
        if cube.dim() == 3:
            cube = cube.unsqueeze(0)
        if cube.shape[0] == 1 and batch_size > 1:
            cube = cube.expand(batch_size, -1, -1, -1)
        return cube

    def forward(self, image, positive_cube, negative_cube):
        bsz = image.shape[0]
        positive_cube = self._broadcast_cube(positive_cube, bsz)
        negative_cube = self._broadcast_cube(negative_cube, bsz)
        ca_maps = {}
        for s in self.scales:
            k = str(s)
            ks, vs = self.key_convs[k](image), self.value_convs[k](image)
            cap = self._attention(ks, vs, self.pos_query_convs[k](positive_cube)) if self.use_positive else torch.zeros_like(vs)
            can = self._attention(ks, vs, self.neg_query_convs[k](negative_cube)) if self.use_negative else torch.zeros_like(vs)
            ca_maps[s] = cap + can
        return ca_maps

class ResidualCrossAttentionWeighting(nn.Module):
    def __init__(self, in_channels=3, nf=15):
        super().__init__()
        self.conv5 = nn.Sequential(_same_pad_conv2d(in_channels, 8, 5), nn.BatchNorm2d(8), nn.GELU())
        self.w5 = nn.Sequential(_same_pad_conv2d(nf, 8, 1), nn.Sigmoid())
        self.conv3 = nn.Sequential(_same_pad_conv2d(8, 16, 3), nn.BatchNorm2d(16), nn.GELU())
        self.w3 = nn.Sequential(_same_pad_conv2d(nf, 16, 1), nn.Sigmoid())
        self.conv1 = nn.Sequential(_same_pad_conv2d(16, 32, 1), nn.BatchNorm2d(32), nn.GELU())
        self.w1 = nn.Sequential(_same_pad_conv2d(nf, 32, 1), nn.Sigmoid())
        self.residual_proj = _same_pad_conv2d(in_channels, 32, 1)
        self.head = nn.Sequential(
            _same_pad_conv2d(32, 32, 3), nn.BatchNorm2d(32), nn.GELU(),
            _same_pad_conv2d(32, 32, 3), nn.BatchNorm2d(32), nn.GELU())

    def forward(self, image, ca_maps):
        if 5 not in ca_maps:
            ca_maps = {1: ca_maps[3], 3: ca_maps[3], 5: ca_maps[3]}
        i5 = self.conv5(image) * self.w5(ca_maps[5])
        i3 = self.conv3(i5) * self.w3(ca_maps[3])
        i1 = self.conv1(i3) * self.w1(ca_maps[1])
        return self.head(i1 + self.residual_proj(image))

class PNMSCA(nn.Module):
    def __init__(self, num_classes, pos_channels, neg_channels, in_channels=3,
                 nf=15, image_size=32, ablation='pnmscca'):
        super().__init__()
        self.num_classes, self.image_size = num_classes, image_size
        self.ablation = ablation.lower()
        scales = (3,) if self.ablation in {'pnssca', 'pssca', 'nssca'} else SCALES
        branches = {
            'pnmscca': (True, True), 'pmscca': (True, False), 'nmscca': (False, True),
            'pnssca': (True, True), 'pssca': (True, False), 'nssca': (False, True),
        }
        use_positive, use_negative = branches[self.ablation]
        self.cross_attention = MultiScaleCrossAttention(
            in_channels, pos_channels, neg_channels, nf, scales, use_positive, use_negative)
        self.residual = ResidualCrossAttentionWeighting(in_channels, nf)
        self.classifier = nn.Linear(32 * image_size * image_size, num_classes)
        self.register_buffer('positive_cube', torch.zeros(1, pos_channels, image_size, image_size))
        self.register_buffer('negative_cube', torch.zeros(1, neg_channels, image_size, image_size))
        self._reps_ready = False

    def set_representatives(self, positive_cube, negative_cube):
        if positive_cube.dim() == 3:
            positive_cube = positive_cube.unsqueeze(0)
        if negative_cube.dim() == 3:
            negative_cube = negative_cube.unsqueeze(0)
        self.positive_cube = positive_cube.contiguous()
        self.negative_cube = negative_cube.contiguous()
        self._reps_ready = True

    def forward(self, images):
        if not self._reps_ready:
            raise RuntimeError('Call set_representatives(P, N) first')
        ca_maps = self.cross_attention(images, self.positive_cube, self.negative_cube)
        return self.classifier(self.residual(images, ca_maps).flatten(1))

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

_m = PNMSCA(10, 15, 15)
_m.set_representatives(torch.rand(1, 15, 32, 32), torch.rand(1, 15, 32, 32))
print('logits', _m(torch.rand(2, 3, 32, 32)).shape, 'params', _m.count_parameters())

logits torch.Size([2, 10]) params 368850


In [15]:
def _safe_div(n, d):
    out = np.zeros_like(n, dtype=np.float64)
    mask = d > 0
    out[mask] = n[mask] / d[mask]
    return out

def compute_classification_metrics(y_true, y_pred, y_prob=None, num_classes=None):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    if num_classes is None:
        num_classes = int(max(y_true.max(), y_pred.max()) + 1)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    tp = np.diag(cm).astype(np.float64)
    fp, fn = cm.sum(axis=0) - tp, cm.sum(axis=1) - tp
    tn = cm.sum() - (tp + fp + fn)
    precision, recall = _safe_div(tp, tp + fp), _safe_div(tp, tp + fn)
    selectivity = _safe_div(tn, tn + fp)
    fmeasure = _safe_div(2 * precision * recall, precision + recall)
    gmean = np.sqrt(np.clip(recall * selectivity, 0.0, None))
    macro_auc = float('nan')
    if y_prob is not None:
        try:
            macro_auc = float(roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro',
                                            labels=list(range(num_classes))))
        except ValueError:
            pass
    return {
        'overall_accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(np.mean(precision)),
        'recall_macro': float(np.mean(recall)),
        'fmeasure_macro': float(np.mean(fmeasure)),
        'selectivity_macro': float(np.mean(selectivity)),
        'gmean_macro': float(np.mean(gmean)),
        'macro_auc': macro_auc,
        'per_class': {
            'precision': precision.tolist(), 'recall': recall.tolist(),
            'fmeasure': fmeasure.tolist(), 'selectivity': selectivity.tolist(),
            'gmean': gmean.tolist(), 'support': cm.sum(axis=1).astype(int).tolist(),
        },
        'confusion_matrix': cm.tolist(),
    }

print('OK: metrics')

OK: metrics


In [16]:
@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    model.eval()
    all_true, all_pred, all_prob = [], [], []
    total_loss, n = 0.0, 0
    criterion = nn.CrossEntropyLoss()
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        n += labels.size(0)
        probs = torch.softmax(logits, dim=1)
        all_true.append(labels.cpu())
        all_pred.append(probs.argmax(dim=1).cpu())
        all_prob.append(probs.cpu())
    metrics = compute_classification_metrics(
        torch.cat(all_true).numpy(), torch.cat(all_pred).numpy(),
        torch.cat(all_prob).numpy(), num_classes=num_classes)
    metrics['loss'] = total_loss / max(1, n)
    return metrics

def train_one_epoch(model, loader, optimizer, criterion, device, epoch):
    model.train()
    running, n = 0.0, 0
    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running += loss.item() * labels.size(0)
        n += labels.size(0)
        pbar.set_postfix(loss=f'{running / max(1, n):.4f}')
    return running / max(1, n)

print('OK: train helpers')

OK: train helpers


In [17]:
train_loader, test_loader, meta, pos_cube, neg_cube = build_dataloaders(
    dataset_name=CONFIG['dataset'],
    data_root=CONFIG['data_root'],
    rho=CONFIG['rho'],
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    seed=CONFIG['seed'],
    fer_csv=CONFIG['fer_csv'],
)
cfg = meta['config']
print(
    f"Dataset={CONFIG['dataset']} train_n={meta['train']['num_samples']} "
    f"test_n={meta['test']['num_samples']} "
    f"rho_train={meta['train']['rho_effective']:.3f} "
    f"rho_test={meta['test']['rho_effective']:.3f}"
)
print('Class counts (train):', meta['train']['counts'].tolist())

model = PNMSCA(
    num_classes=meta['num_classes'],
    pos_channels=meta['pos_channels'],
    neg_channels=meta['neg_channels'],
    nf=CONFIG['nf'],
    image_size=meta['image_size'],
    ablation=CONFIG['ablation'],
)
model.set_representatives(pos_cube, neg_cube)
model = model.to(DEVICE)
print('Trainable parameters:', f'{model.count_parameters():,}')

Dataset=cifar10 train_n=27500 test_n=5500 rho_train=10.000 rho_test=10.000
Class counts (train): [500, 500, 500, 500, 500, 5000, 5000, 5000, 5000, 5000]
Trainable parameters: 368,850


In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=CONFIG['lr'])
ckpt_dir = Path(CONFIG['checkpoint_dir'])
ckpt_dir.mkdir(parents=True, exist_ok=True)
run_name = (
    f"{CONFIG['dataset']}_{CONFIG['ablation']}_"
    f"rho{meta['train']['rho_effective']:.2f}_s{CONFIG['seed']}"
)
best_path = ckpt_dir / f'{run_name}_best.pt'
history, best_f1 = [], -1.0
t0 = time.time()

for epoch in range(1, CONFIG['epochs'] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, epoch)
    metrics = evaluate(model, test_loader, DEVICE, num_classes=meta['num_classes'])
    row = {
        'epoch': epoch, 'train_loss': train_loss, 'test_loss': metrics['loss'],
        'overall_accuracy': metrics['overall_accuracy'],
        'precision_macro': metrics['precision_macro'],
        'recall_macro': metrics['recall_macro'],
        'fmeasure_macro': metrics['fmeasure_macro'],
        'selectivity_macro': metrics['selectivity_macro'],
        'gmean_macro': metrics['gmean_macro'],
        'macro_auc': metrics['macro_auc'],
    }
    history.append(row)
    print(
        f"[{epoch:02d}/{CONFIG['epochs']}] train_loss={train_loss:.4f} "
        f"acc={metrics['overall_accuracy']:.4f} F1={metrics['fmeasure_macro']:.4f} "
        f"Gmean={metrics['gmean_macro']:.4f} AUC={metrics['macro_auc']:.4f}"
    )
    if metrics['fmeasure_macro'] > best_f1:
        best_f1 = metrics['fmeasure_macro']
        torch.save({
            'model_state': model.state_dict(),
            'config': CONFIG,
            'meta': {
                'num_classes': meta['num_classes'], 'image_size': meta['image_size'],
                'pos_channels': meta['pos_channels'], 'neg_channels': meta['neg_channels'],
                'train_counts': meta['train']['counts'].tolist(),
                'test_counts': meta['test']['counts'].tolist(),
                'rho_train': meta['train']['rho_effective'],
                'rho_test': meta['test']['rho_effective'],
                'minority_classes': cfg['minority_classes'],
                'majority_classes': cfg['majority_classes'],
            },
            'metrics': {k: v for k, v in metrics.items() if k != 'confusion_matrix'},
            'positive_cube': pos_cube.cpu(),
            'negative_cube': neg_cube.cpu(),
        }, best_path)

elapsed = time.time() - t0
final_metrics = evaluate(model, test_loader, DEVICE, num_classes=meta['num_classes'])
results = {
    'run_name': run_name, 'elapsed_sec': elapsed, 'best_fmeasure_macro': best_f1,
    'final_metrics': {k: v for k, v in final_metrics.items() if k != 'confusion_matrix'},
    'history': history, 'checkpoint': str(best_path),
}
out_json = ckpt_dir / f'{run_name}_results.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
print(f'Done in {elapsed:.1f}s | best F1={best_f1:.4f}')
print('Saved:', best_path)
print('Saved:', out_json)

Epoch 1:   0%|          | 0/27 [00:00<?, ?it/s]

[01/25] train_loss=2.3961 acc=0.3784 F1=0.1620 Gmean=0.2450 AUC=0.7535


Epoch 2:   0%|          | 0/27 [00:00<?, ?it/s]

[02/25] train_loss=1.1039 acc=0.6198 F1=0.3455 Gmean=0.4535 AUC=0.8729


Epoch 3:   0%|          | 0/27 [00:00<?, ?it/s]

[03/25] train_loss=0.9155 acc=0.6818 F1=0.4283 Gmean=0.5566 AUC=0.9100


... (40 outputs left out)

Epoch 24:   0%|          | 0/27 [00:00<?, ?it/s]

[24/25] train_loss=0.0119 acc=0.7704 F1=0.5159 Gmean=0.6397 AUC=0.9329


Epoch 25:   0%|          | 0/27 [00:00<?, ?it/s]

[25/25] train_loss=0.0112 acc=0.7687 F1=0.5099 Gmean=0.6341 AUC=0.9333
Done in 347.6s | best F1=0.5162
Saved: checkpoints/cifar10_pnmscca_rho10.00_s0_best.pt
Saved: checkpoints/cifar10_pnmscca_rho10.00_s0_results.json


In [19]:
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
eval_model = PNMSCA(
    num_classes=ckpt['meta']['num_classes'],
    pos_channels=ckpt['positive_cube'].shape[1],
    neg_channels=ckpt['negative_cube'].shape[1],
    nf=ckpt['config'].get('nf', 15),
    image_size=ckpt['meta']['image_size'],
    ablation=ckpt['config'].get('ablation', 'pnmscca'),
)
eval_model.set_representatives(ckpt['positive_cube'], ckpt['negative_cube'])
eval_model.load_state_dict(ckpt['model_state'])
eval_model = eval_model.to(DEVICE)
eval_metrics = evaluate(eval_model, test_loader, DEVICE, num_classes=ckpt['meta']['num_classes'])
for k in ['overall_accuracy', 'precision_macro', 'recall_macro', 'fmeasure_macro',
          'selectivity_macro', 'gmean_macro', 'macro_auc', 'loss']:
    print(f'{k}: {eval_metrics[k]:.4f}')

overall_accuracy: 0.7709
precision_macro: 0.5974
recall_macro: 0.4960
fmeasure_macro: 0.5162
selectivity_macro: 0.9724
gmean_macro: 0.6385
macro_auc: 0.9323
loss: 0.9690
